In [29]:
import os
import json
import numpy as np

In [ ]:
# TODO: make a class for each video?
def get_video_metadata(video_dir):
    gesture_dirs = [d for d in os.listdir(video_dir) if os.path.isdir(os.path.join(video_dir, d))]
    
    for gesture_name in gesture_dirs:
        images_path = os.path.join(video_dir, gesture_name, "images")
        # Safely check if images_path contains any files
        if not os.path.exists(images_path): print(f"{images_path} does not exist"); continue
        image_dirs = [
            f
            for f in os.listdir(images_path)
        ]
        metadata_files = [os.path.join(images_path, d, "metadata.json") for d in image_dirs if d.startswith(f"{gesture_name}")]
        # metadata_path = os.path.join(video_dir, gesture_name, "images/metadata.json")
        # print(f"Reading {metadata_path}")
        print(f"{gesture_name}: {len(image_dirs)} image files")
        print(metadata_files)
        return metadata_files

In [23]:
metadata_files = get_video_metadata("/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align")

paper: 11 image files
['/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_49/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_38/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_36/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_31/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_37/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_45/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_20/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_35/metadata.json', '/Users/christina/code/RockPaperScissors/my_rps_dataset/data/align/paper/images/paper_50/metadata.json', '/Users/christina/code/RockPaper

In [27]:
def get_cropped_video_indices(video_metadata: dict):
    start, onset, end = video_metadata["start_frame_idx"], video_metadata["action_frame_idx"], video_metadata["end_frame_idx"]
    return start, onset, end

Check landmarks at cropeed video frames

In [45]:
def get_overlapping_chunks(features, chunk_size=5, overlap=1):
    """Get overlapping chunks of the features (x[0:chunk_size], x[1:chunk_size+1]) along the first dimension"""
    step = chunk_size - overlap
    chunks = []
    idx = []
    # Assume features is (T, D, 3)
    n_frames = features.shape[0]
    for start in range(0, n_frames - chunk_size + 1, overlap):
        chunk = features[start : start + chunk_size]
        # print((start, start + chunk_size), overlap)
        chunks.append(chunk)
        idx.append((start, start + chunk_size))
    chunks.append(features[start:])
    idx.append((start + chunk_size, n_frames))
    return chunks, idx

In [53]:
from scipy.spatial.distance import pdist

LANDMARKS_DIR = "/Users/christina/code/RockPaperScissors/my_rps_dataset/features_coords"
new_features = {}
for meta_file in metadata_files:

    with open(meta_file, "r") as f:
        data = json.load(f)

    start, onset, end = get_cropped_video_indices(data)
    # print(f"Start: {start}, Onset: {onset}, End: {end}")
    vid_name = meta_file.split('/')[-2]
    landmarks_file = os.path.join(LANDMARKS_DIR, f"{vid_name}_landmarks.npz")
    landmarks_data = np.load(landmarks_file, allow_pickle=True)
    landmarks = landmarks_data["landmarks"][start:end+1]
    print(f"Number of NaNs: {np.isnan(landmarks).sum()}")
    
    CHUNK_SIZE, STRIDE = 3, 1
    chunks, idx = get_overlapping_chunks(
        landmarks, chunk_size=CHUNK_SIZE, overlap=STRIDE
    )
    sample_seq = []
    for i, (c, (start, end)) in enumerate(zip(chunks, idx)):
        if c.shape[0] < CHUNK_SIZE:
            print("  Skipping, too short")
            continue
        else:
            sample_seq.append(c)
    new_features[vid_name] = np.array(sample_seq)

    # Compute pairwise distances for each chunk index across samples
    N_CHUNKS = new_features[vid_name].shape[0]
    res = []
    res_var = []

    for k in range(N_CHUNKS):
        chunks_at_k = []

        for sample_id, sample_seq in new_features.items():
            chunks_at_k.append(sample_seq[k])  # shape (5, D) where D can be 63 for coords and 5 for angles

        # Stack to get (n_samples, 5, D)
        chunks_at_k = np.stack(chunks_at_k, axis=0)
        
        # Compute pairwise distances between all samples at each time step
        N_SLIDING_WINDOWS = chunks_at_k.shape[1]
        dist = np.array(
            [
                pdist(chunks_at_k[:, i, :], metric="euclidean")
                for i in range(N_SLIDING_WINDOWS)
            ]
        )

        res.append(dist.mean())
        # compute standard error of the mean across samples
        res_var.append(dist.std() / np.sqrt(dist.shape[0]))


Number of NaNs: 0


ValueError: A 2-dimensional array must be passed. (Shape was (1, 21, 3)).

In [55]:
sample_seq[k].shape

(3, 21, 3)

In [43]:
landmarks

array([[ 2.47836247e-01,  5.65443039e-01, -2.15461597e-07,
         3.05604637e-01,  4.75023568e-01, -9.55334771e-03,
         3.88269186e-01,  4.32653397e-01, -2.88000349e-02,
         4.60583776e-01,  4.37828362e-01, -4.66548167e-02,
         4.89299327e-01,  4.72797453e-01, -6.05286621e-02,
         4.19250607e-01,  4.34274077e-01, -4.50440384e-02,
         4.92446661e-01,  5.00463367e-01, -6.61107525e-02,
         4.62727070e-01,  5.08539736e-01, -7.45015517e-02,
         4.31784838e-01,  4.93910939e-01, -7.94645622e-02,
         4.06757325e-01,  5.08544326e-01, -5.52384220e-02,
         4.86934572e-01,  5.64562976e-01, -7.00749457e-02,
         4.54786628e-01,  5.71342707e-01, -6.70551807e-02,
         4.18851584e-01,  5.64245343e-01, -6.52288571e-02,
         3.92030180e-01,  5.85469067e-01, -6.41352013e-02,
         4.69270408e-01,  6.26778603e-01, -7.42513165e-02,
         4.36122030e-01,  6.30790055e-01, -5.67164645e-02,
         3.98571849e-01,  6.21870100e-01, -4.50517200e-0